# 1. Raw CMOS scans → photon-filtered averages
Run top to bottom. One scan index selects **all detector files and all frames** in that scan; each index produces one average. Dark scans are pooled by frame count. Filtering follows ajajas: subtract the dark from each frame, zero signal at or below the threshold, retain analog signal above it, then average. It does not turn ADU into integer photon counts.

Edit the paths, scan indices, energies, and polarizations below. Raw data stay on the raw detector grid. All numerical results and the first example frame are saved to HDF5.
Nonfinite and saturated raw samples become NaN and are ignored when averaging. A pixel with no valid samples remains NaN and is excluded by notebook 2. Photon-thresholded samples remain valid zeros.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
if HERE.name != "permanent_magnets":
    HERE = HERE / "permanent_magnets"
REPO = HERE.parent
sys.path.insert(0, str(REPO))
from permanent_magnets import workflow


In [ ]:
PROCESSED = Path("/path/to/processed")  # Edit: same output folder in all four notebooks.
RAW_ROOT = Path("/path/to/raw")  # Edit: folder containing the scan directories.
PREFIX = "scan"  # Edit: your scan-directory prefix.
SCAN_INDICES = [1, 2]  # One output observation per whole scan.
DARK_INDICES = [0]       # [] explicitly disables dark subtraction.
ENERGIES_EV = [781.0, 781.0]  # One value per output observation; check experiment metadata.
STATE_LABELS = ["state_0", "state_0"]  # Same label for observations of the same magnetic state.
POLARIZATIONS = [1.0, -1.0]   # Helicity coefficients; use -1 for opposite helicity.
PHOTON_THRESHOLD = 6.0       # ADU after dark subtraction; None disables filtering.
SATURATION = 65535  # Raw ADU at or above this value become NaN and are ignored.
INVALID_PIXEL_POLICY = "all_frames"  # "all_frames": mask only if no valid samples; "any_frame": mask if any sample is invalid.
DATASET = "/entry/instrument/detector/data"  # HDF5 dataset containing the raw CMOS frames.
OUTPUT = PROCESSED / "01_averages.h5"  # HDF5 file written by this notebook; overwritten on rerun.
assert len(SCAN_INDICES) == len(ENERGIES_EV) == len(POLARIZATIONS) == len(STATE_LABELS)
assert len(SCAN_INDICES) > 0 and len(set(SCAN_INDICES)) == len(SCAN_INDICES)
assert np.all(np.isfinite(ENERGIES_EV)) and np.all(np.asarray(ENERGIES_EV) > 0)
assert np.all(np.isfinite(POLARIZATIONS))
assert PHOTON_THRESHOLD is None or np.isfinite(PHOTON_THRESHOLD)

assert INVALID_PIXEL_POLICY in ("all_frames", "any_frame")


## Load, filter, average, and save


In [ ]:
sample_files = [workflow.scan_paths(RAW_ROOT, PREFIX, index) for index in SCAN_INDICES]
dark_files = [path for index in DARK_INDICES for path in workflow.scan_paths(RAW_ROOT, PREFIX, index)]
with h5py.File(sample_files[0][0], "r") as h:
    shape = h[DATASET].shape[-2:]

# Streaming nanmean of dark frames: sum finite samples and count them per pixel.
dark = np.zeros(shape, dtype=float)
dark_count = 0
dark_valid_counts = np.zeros(shape, dtype=np.uint32)
for path in dark_files:
    with h5py.File(path, "r") as h:
        data = h[DATASET]
        assert data.ndim in (2, 3) and data.shape[-2:] == shape
        for j in range(1 if data.ndim == 2 else len(data)):
            frame = np.asarray(data[()] if data.ndim == 2 else data[j], dtype=float)
            frame = np.where(np.isfinite(frame) & (frame < SATURATION), frame, np.nan)
            dark += np.nan_to_num(frame, nan=0.)
            dark_valid_counts += np.isfinite(frame)
            dark_count += 1
if dark_count:
    dark = np.divide(dark, dark_valid_counts, out=np.full(shape, np.nan), where=dark_valid_counts > 0)
    if INVALID_PIXEL_POLICY == "any_frame":
        dark[dark_valid_counts < dark_count] = np.nan
print("Dark frames:", dark_count)

# One average per scan: read a frame, filter it, add it to the sum.
images = np.zeros((len(SCAN_INDICES), *shape), dtype=np.float32)
frame_counts = np.zeros(len(SCAN_INDICES), dtype=int)
valid_counts = np.zeros(images.shape, dtype=np.uint32)  # Valid exposure count for each pixel.
for i, paths in enumerate(sample_files):
    total = np.zeros(shape, dtype=float)
    for path in paths:
        with h5py.File(path, "r") as h:
            data = h[DATASET]
            assert data.ndim in (2, 3) and data.shape[-2:] == shape
            for j in range(1 if data.ndim == 2 else len(data)):
                raw = np.asarray(data[()] if data.ndim == 2 else data[j], dtype=float)
                frame = np.where(np.isfinite(raw) & (raw < SATURATION), raw - dark, np.nan)
                if PHOTON_THRESHOLD is not None:
                    frame[frame <= PHOTON_THRESHOLD] = 0
                total += np.nan_to_num(frame, nan=0.)
                valid_counts[i] += np.isfinite(frame)
                frame_counts[i] += 1
                if i == 0 and frame_counts[i] == 1:
                    raw_example = raw.copy()
                    filtered_example = frame.copy()
    if frame_counts[i] == 0:
        raise ValueError(f"No frames in scan {SCAN_INDICES[i]}")
    # Equivalent to np.nanmean across all frames, without storing the raw stack.
    images[i] = np.divide(total, valid_counts[i], out=np.full(shape, np.nan), where=valid_counts[i] > 0)
    if INVALID_PIXEL_POLICY == "any_frame":
        images[i][valid_counts[i] < frame_counts[i]] = np.nan
    print(f"Scan {SCAN_INDICES[i]}: {len(paths)} files, {frame_counts[i]} frames")

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT, "w") as h:
    h.attrs["raw_root"] = str(RAW_ROOT)
    h.attrs["prefix"] = PREFIX
    h.attrs["detector_dataset"] = DATASET
    h.attrs["photon_threshold_ADU"] = np.nan if PHOTON_THRESHOLD is None else PHOTON_THRESHOLD
    h.attrs["saturation_ADU"] = SATURATION
    h.attrs["filter_rule"] = "nonfinite/saturated samples -> NaN; dark subtraction; threshold; nanmean"
    h.attrs["invalid_pixel_policy"] = INVALID_PIXEL_POLICY
    h.attrs["dark_frames"] = dark_count
    h.create_dataset("averages", data=images, compression="gzip")
    h.create_dataset("dark", data=dark, compression="gzip")
    h.create_dataset("frame_counts", data=frame_counts)
    h.create_dataset("valid_counts", data=valid_counts, compression="gzip")
    h.create_dataset("dark_valid_counts", data=dark_valid_counts, compression="gzip")
    h.create_dataset("scan_indices", data=SCAN_INDICES)
    h.create_dataset("dark_indices", data=DARK_INDICES)
    h.create_dataset("energy_eV", data=ENERGIES_EV)
    h.create_dataset("polarization", data=POLARIZATIONS)
    h.create_dataset("state_labels", data=STATE_LABELS, dtype=h5py.string_dtype())
    h.create_dataset("example_raw", data=raw_example, compression="gzip")
    h.create_dataset("example_filtered", data=filtered_example, compression="gzip")
    sources = h.create_group("source_files")
    sources.create_dataset("dark", data=[str(p) for p in dark_files], dtype=h5py.string_dtype())
    for i, paths in enumerate(sample_files):
        sources.create_dataset(str(i), data=[str(p) for p in paths], dtype=h5py.string_dtype())
print("Saved", OUTPUT)


## Inspect the threshold and the averages

In [ ]:
with h5py.File(OUTPUT, "r") as h:
    images = h["averages"][()]
    raw_example = h["example_raw"][()]
    filtered_example = h["example_filtered"][()]
    frame_counts = h["frame_counts"][()]
corrected_example = raw_example - dark
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(dark); axes[0].set_title("Mean dark (ADU)")
axes[1].hist(corrected_example[np.isfinite(corrected_example)].ravel(), bins=300, range=(-20, 100), log=True)
if PHOTON_THRESHOLD is not None:
    axes[1].axvline(PHOTON_THRESHOLD, color="red", label="Threshold"); axes[1].legend()
axes[1].set(xlabel="Dark-corrected ADU", ylabel="Pixels", title="First-frame threshold diagnostic")
axes[2].imshow(np.log1p(np.maximum(filtered_example, 0))); axes[2].set_title("Filtered example frame, log(1+ADU)")
plt.show()
workflow.plot_stack(np.log1p(np.maximum(images, 0)), SCAN_INDICES, "Filtered averages, log(1+ADU)")
print("Frames per average:", frame_counts)